<a href="https://colab.research.google.com/github/Disha092006/Flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Disha092006/Flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule in plain words

I will prioritize content that has meaningful search visibility but a weak click-through rate relative to its search position. Higher impressions increase the priority because the page has more opportunity to benefit from an improvement.

### Reason code

- `ctr_fix_opportunity` — the page has enough search visibility, but its CTR is low for its observed average position.

### Action label

- `review_ctr` — review the title/snippet and other search-result presentation factors before making changes.

In [3]:
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import hf_hub_download

# Get Hugging Face token directly from Colab Secrets
token = userdata.get("HF_TOKEN")

# Load March 2026
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

march_df = pd.read_parquet(file_path)

print("March 2026 rows:", len(march_df))

# -----------------------------
# Signal 1: CTR vs position
# -----------------------------

signal_df = march_df[
    ["client_hash_id", "content_hash_id",
     "gsc_impressions", "gsc_clicks", "gsc_avg_position"]
].copy()

signal_df["ctr"] = np.where(
    signal_df["gsc_impressions"] > 0,
    signal_df["gsc_clicks"] / signal_df["gsc_impressions"],
    np.nan
)

signal_df["position_bucket"] = pd.cut(
    signal_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

ctr_position_table = (
    signal_df
    .dropna(subset=["position_bucket", "ctr"])
    .groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 1 — CTR vs POSITION")
print(ctr_position_table.to_string(index=False))


# -----------------------------
# Signal 2: Search volume
# -----------------------------

volume_df = signal_df[
    signal_df["gsc_impressions"] > 0
].copy()

volume_df["volume_bucket"] = pd.qcut(
    volume_df["gsc_impressions"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

volume_table = (
    volume_df
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("gsc_impressions", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_clicks=("gsc_clicks", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — SEARCH VOLUME")
print(volume_table.to_string(index=False))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March 2026 rows: 9841378

SIGNAL 1 — CTR vs POSITION
position_bucket       n  median_ctr  mean_ctr
            1-3  727362         0.0  0.004756
           4-10 1456122         0.0  0.003473
          11-20  519223         0.0  0.002770
            21+  908354         0.0  0.001289

SIGNAL 2 — SEARCH VOLUME
volume_bucket       n  median_impressions  median_clicks
          Low 1209356                 2.0            0.0
       Medium 1205478                16.0            0.0
         High 1196227               108.0            0.0


### Signal checks and verdicts

**1. CTR vs position — CONFIRMED**

The observed mean CTR decreases as average position moves from 1–3 to 21+. This supports using CTR relative to search position as a directional signal for identifying possible CTR-fix opportunities. The median CTR is 0 in every bucket, so the relationship should be treated as directional rather than as a precise benchmark.

**2. Search volume — MIXED**

Impressions increase substantially from the Low to High volume buckets, showing that higher-volume content has more search exposure. However, median clicks are 0 in all three buckets, so volume alone does not demonstrate stronger click performance. I will therefore use volume as an opportunity/priority signal rather than as evidence that a page will improve.

### Baseline rule

The score combines two observed March signals: search exposure and CTR weakness relative to the observed position bucket.

Higher impressions increase priority because they represent greater search exposure. A larger positive gap between the observed CTR and the mean CTR for the same position bucket increases priority for a possible CTR review.

Each ranked row receives one reason code, `ctr_fix_opportunity`, and the action label `review_ctr`. The rule uses only March decision-time observations and does not use future outcomes or label-derived fields.

In [4]:
# Section 2 — Build the ranked baseline queue

# Start from the March decision-time data already loaded.
baseline_df = signal_df.copy()

# Keep only rows where the signals needed by the rule are usable.
baseline_df = baseline_df[
    (baseline_df["gsc_impressions"] > 0) &
    (baseline_df["gsc_avg_position"].notna()) &
    (baseline_df["gsc_avg_position"] > 0)
].copy()

# Score:
# - Higher impressions = more opportunity
# - Lower CTR for a given position = stronger CTR-fix opportunity
#
# Expected CTR is estimated within each position bucket.
baseline_df["position_bucket"] = pd.cut(
    baseline_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

expected_ctr = (
    baseline_df
    .groupby("position_bucket", observed=True)["ctr"]
    .mean()
    .rename("expected_ctr")
)

baseline_df = baseline_df.join(expected_ctr, on="position_bucket")

# CTR gap: larger positive value means CTR is below the observed
# average for its position bucket.
baseline_df["ctr_gap"] = (
    baseline_df["expected_ctr"] - baseline_df["ctr"]
).clip(lower=0)

# Normalize the two components so they contribute on comparable scales.
baseline_df["volume_score"] = np.log1p(
    baseline_df["gsc_impressions"]
)

baseline_df["ctr_gap_score"] = (
    baseline_df["ctr_gap"] * 100
)

# Final transparent score.
baseline_df["action_score"] = (
    baseline_df["volume_score"] * baseline_df["ctr_gap_score"]
)

# One reason code and one action label.
baseline_df["reason_code"] = np.where(
    baseline_df["action_score"] > 0,
    "ctr_fix_opportunity",
    "no_action"
)

baseline_df["action"] = np.where(
    baseline_df["action_score"] > 0,
    "review_ctr",
    "no_action"
)

# Rank highest-priority opportunities first.
baseline_df = baseline_df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

baseline_df["rank"] = np.arange(1, len(baseline_df) + 1)

# Keep the ranked queue compact and auditable.
queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "expected_ctr",
    "ctr_gap",
    "action_score",
    "reason_code",
    "action"
]

baseline_queue = baseline_df[queue_columns].copy()

# Write the required CSV.
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(output_path, index=False)

print("Rows in ranked queue:", len(baseline_queue))
print("CSV written to:", output_path)

print("\nTop 10:")
print(baseline_queue.head(10).to_string(index=False))

Rows in ranked queue: 3447872
CSV written to: work/outputs/baseline_action_score.csv

Top 10:
 rank          client_hash_id          content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position      ctr  expected_ctr  ctr_gap  action_score         reason_code     action
    1 client_23a62021009f63c4 content_44f34c0a90047651            40084           1          0.083350 0.000025      0.004918 0.004893      5.186248 ctr_fix_opportunity review_ctr
    2 client_62f4a7e64f5e0096 content_34a70fea29d15f24            39003           2          2.764916 0.000051      0.004918 0.004867      5.145035 ctr_fix_opportunity review_ctr
    3 client_73cda7b4e4f265ea content_fec55986a1868d62            33383           0          0.181500 0.000000      0.004918 0.004918      5.122723 ctr_fix_opportunity review_ctr
    4 client_23a62021009f63c4 content_44f34c0a90047651            32958           0          0.132532 0.000000      0.004918 0.004918      5.116421 ctr_fix_opportunity review_ctr
    5 clien

### Top-20 review

I reviewed the highest-ranked rows individually. Each review records the action, the reason code, a confidence note based on the observed March signals, and what could make the recommendation wrong.

The review is deliberately skeptical: a high score indicates a measured opportunity under this baseline rule, not proof that changing the page will improve performance.

In [5]:
# Section 3 — Top-20 review

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    if row["ctr_gap"] >= 0.004:
        return "Strong observed CTR gap"
    elif row["ctr_gap"] >= 0.002:
        return "Moderate observed CTR gap"
    else:
        return "Small observed CTR gap"

def what_would_make_it_wrong(row):
    if row["gsc_impressions"] < 100:
        return "Low exposure could make the signal unstable."
    elif row["gsc_clicks"] == 0:
        return "Zero clicks may reflect missing context or unusually low engagement."
    elif row["gsc_avg_position"] <= 1:
        return "Very strong position may make the CTR comparison less reliable."
    else:
        return "The position bucket average may not represent this specific query mix."

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns].copy()

print("TOP-20 REVIEW")
print(top20_review.to_string(index=False))

TOP-20 REVIEW
 rank          client_hash_id          content_hash_id     action         reason_code         confidence_note                                               what_would_make_it_wrong
    1 client_23a62021009f63c4 content_44f34c0a90047651 review_ctr ctr_fix_opportunity Strong observed CTR gap        Very strong position may make the CTR comparison less reliable.
    2 client_62f4a7e64f5e0096 content_34a70fea29d15f24 review_ctr ctr_fix_opportunity Strong observed CTR gap The position bucket average may not represent this specific query mix.
    3 client_73cda7b4e4f265ea content_fec55986a1868d62 review_ctr ctr_fix_opportunity Strong observed CTR gap   Zero clicks may reflect missing context or unusually low engagement.
    4 client_23a62021009f63c4 content_44f34c0a90047651 review_ctr ctr_fix_opportunity Strong observed CTR gap   Zero clicks may reflect missing context or unusually low engagement.
    5 client_73cda7b4e4f265ea content_fec55986a1868d62 review_ctr ctr_fix_opportu

### Weak picks + leakage check

Some picks are weak because the same content item appears on multiple daily rows, so the queue can concentrate several ranks on one content item. Also, the position-bucket comparison is only a directional baseline and may not capture differences in query mix or search intent.

I therefore treat the ranked rows as decision-support opportunities rather than confirmed fixes.

The baseline uses only March 2026 observed fields. No April outcome, future-window measurement, or label-derived field is used in the score or action rule.

In [6]:
# Section 4 — Weak picks + leakage check

# Check that no future/label-derived fields entered the baseline queue.
forbidden_fields = {
    "is_declining_label",
    "april_impressions",
    "april_clicks"
}

leaked_fields = forbidden_fields.intersection(
    set(baseline_queue.columns)
)

print("Forbidden future/label-derived fields found:",
      leaked_fields)

print("\nLeakage check passed:",
      len(leaked_fields) == 0)

# Check concentration of the top 20.
print("\nUnique content items in Top-20:",
      top20["content_hash_id"].nunique())

print("Top-20 rows:",
      len(top20))

print("\nRepeated content items in Top-20:")
print(
    top20["content_hash_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

Forbidden future/label-derived fields found: set()

Leakage check passed: True

Unique content items in Top-20: 7
Top-20 rows: 20

Repeated content items in Top-20:
content_hash_id
content_44f34c0a90047651    6
content_8e1334d6356668e3    4
content_fec55986a1868d62    3
content_9c057b66c30a3abb    3
content_8d7d99f109e19aa2    2
Name: count, dtype: int64


## Self-check

- Every section is filled with both Markdown reasoning and supporting code.
- The notebook runs top to bottom without errors.
- Two signal checks have visible bucket tables with `n` values and one-word verdicts.
- At least one checked signal is linked to a documented FlyRank flag logic: CTR vs position.
- One transparent baseline rule produces a score, one reason code, and one action label.
- The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- The Top-20 rows have an action, reason code, confidence note, and what would make each pick wrong.
- Weak picks are documented, including repeated content items in the Top-20.
- The leakage check found no future-window or label-derived fields in the baseline queue.
- No client names, URLs, or private queries are included.
- Claims are framed as observed, measured, directional, or decision-support rather than guaranteed outcomes.